# ИЗ-2. NumPy: векторные операции над массивами

**Евстафьева Вера, ИТ-14-2024, вариант 06**.

Перед сдачей: **Kernel → Restart & Run All**. Работа, которая так не проходит,
не проверяется. Выводы ячеек должны остаться в файле — ноутбук с пустыми
выводами смотреть не в чем.

Файл называется `ИЗ-2_Евстафьева_Вера_вариант06.ipynb`, сдача через Moodle.


## Настройка

In [1]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SEED = 42                      # одно зерно на всю работу
np.random.seed(SEED)
pd.set_option("display.width", 110)
pd.set_option("display.max_columns", 20)

df = pd.read_csv("data.csv")    # файл лежит рядом с ноутбуком
df.shape


(2485, 20)

## ● Пункт 1. Профиль данных и пропуски

<sub>обязательный пункт · 1 балл</sub>

In [2]:
# --- из таблицы из настроек в матрицу Х ---
num_names = [col for col in df.columns if col not in ['row_id', 'y_clf']]
X = df[num_names].values
cat = {"waterfront": df["waterfront"].values, "view": df["view"].values, "y_clf": df["y_clf"].values}
ids = ["row_id"]

n, m = X.shape

miss = np.isnan(X).sum(axis=0)
miss_share = miss / n
mn = np.nanmin(X, axis=0)
mx = np.nanmax(X, axis=0)
mean = np.nanmean(X, axis=0)
median = np.nanmedian(X, axis=0)
sd = np.nanstd(X, axis=0, ddof=1)

header = f"{'столбец':<16}{'пропуски':>7}{'доля':>8}{'min':>12}{'max':>12}{'среднее':>12}{'медиана':>12}{'СКО':>12}"
print(header)
print("-" * len(header))
for j, name in enumerate(num_names):
    print(f"{name:<16}{miss[j]:>7d}{miss_share[j]:>8.2%}"
          f"{mn[j]:>12.3f}{mx[j]:>12.3f}{mean[j]:>12.3f}{median[j]:>12.3f}{sd[j]:>12.3f}")


столбец         пропуски    доля         min         max     среднее     медиана         СКО
--------------------------------------------------------------------------------------------
price                 0   0.00%   75000.000 4500000.000  506239.819  450000.000  272260.956
bedrooms              0   0.00%      -2.000      11.000       3.320       3.000       0.984
bathrooms             0   0.00%       0.000       7.500       2.019       2.000       0.759
sqft_living           0   0.00%     470.000    7220.000    1983.668    1820.000     863.020
sqft_lot              0   0.00%     572.000 1651359.000   17658.165    7920.000   57498.168
floors                0   0.00%       1.000       3.500       1.442       1.000       0.524
waterfront            0   0.00%       0.000       1.000       0.005       0.000       0.069
view                  0   0.00%       0.000       4.000       0.179       0.000       0.669
grade                 0   0.00%       3.000      12.000       7.514       7.00

**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Первая аномалия: Столбец sqft_basement выделяется долей пропусков: 290 (11.67%), тогда как у всех остальных столбцов пропусков нет, а его медиана равна 0 при среднем 283.4. Это означает, что данные в этом столбце собирались иначе, чем в остальных, и при дальнейшем анализе его нельзя рассматривать наравне с полностью заполненными признаками — нужно отдельно разбираться, почему там возникают пустые клетки;
Вторая аномалия: В столбце bedrooms минимальное значение -2 (отрицательное число спален), это явная опечатка или ошибка ввода.

## ● Пункт 2. Приведение к общей шкале

<sub>обязательный пункт · 1 балл</sub>

In [3]:
# Выкидываю столбец с пропусками
j_base = num_names.index("sqft_basement")
X_clean = np.delete(X, j_base, axis=1)
names_clean = np.delete(num_names, j_base)

# Стандартная минимаксная нормализация
mn = X_clean.min(axis=0)
mx = X_clean.max(axis=0)
den = np.where(mx - mn == 0, 1.0, mx - mn)
Xminmax = (X_clean - mn) / den

# Стандартизация (z-оценки)
mu = X_clean.mean(axis=0)
sd = X_clean.std(axis=0, ddof=1)
sd = np.where(sd == 0, 1.0, sd)
Xz = (X_clean - mu) / sd

print("Данные после min-max:")
print("  min:  ", np.round(Xminmax.min(axis=0), 4)[:5], "...")
print("  max:  ", np.round(Xminmax.max(axis=0), 4)[:5], "...")
print("  mean: ", np.round(Xminmax.mean(axis=0), 4)[:5], "...")
print("  std:  ", np.round(Xminmax.std(axis=0, ddof=1), 4)[:5], "...")

print("\nДанные после z-оценок:")
print("  min:  ", np.round(Xz.min(axis=0), 4)[:5], "...")
print("  max:  ", np.round(Xz.max(axis=0), 4)[:5], "...")
print("  mean: ", np.round(Xz.mean(axis=0), 4)[:5], "...")
print("  std:  ", np.round(Xz.std(axis=0, ddof=1), 4)[:5], "...")

# Столбец с самым большим разрывом между max и p95
p95 = np.nanpercentile(X_clean, 95, axis=0)
gap = mx - p95
j = np.argmax(gap)
target_name = names_clean[j]

print(f"\nАнализ столбца: {target_name}")
print(f"Данные до:")
print(f"  max = {mx[j]:.1f}")
print(f"  p95 = {p95[j]:.1f}")
print(f"  разрыв = {gap[j]:.1f}")

print(f"\nДанные после Min-Max:")
print(f"  max: {Xminmax[:, j].max():.4f}")
p95_minmax = np.percentile(Xminmax[:, j], 95)
print(f"  p95: {p95_minmax:.4f}")

print(f"\nДанные после Z-оценки:")
print(f"  max: {Xz[:, j].max():.2f}")
p95_z = np.percentile(Xz[:, j], 95)
print(f"  p95: {p95_z:.2f}")

Данные после min-max:
  min:   [0. 0. 0. 0. 0.] ...
  max:   [1. 1. 1. 1. 1.] ...
  mean:  [0.0975 0.4093 0.2692 0.2242 0.0104] ...
  std:   [0.0615 0.0757 0.1012 0.1279 0.0348] ...

Данные после z-оценок:
  min:   [-1.5839 -5.4047 -2.6611 -1.7539 -0.2972] ...
  max:   [14.6689  7.8015  7.224   6.0675 28.4131] ...
  mean:  [ 0. -0.  0.  0. -0.] ...
  std:   [1. 1. 1. 1. 1.] ...

Анализ столбца: price
Данные до:
  max = 4500000.0
  p95 = 907000.0
  разрыв = 3593000.0

Данные после Min-Max:
  max: 1.0000
  p95: 0.1880

Данные после Z-оценки:
  max: 14.67
  p95: 1.47


**Ответ.** _Здесь письменный ответ на вопрос пункта._

> Столбец с самым большим разрывом между максимумом и 95-м перцентилем - price: максимум равен 4 500 000, а 95-й перцентиль - 907 000, то есть разрыв составляет 3 593 000. Минимаксная шкала сжала этот столбец в [0, 1], и после неё 95% цен оказались ниже 0.1880 - основная масса данных «сплющилась» в узкую полосу от нуля до 0.19, а всё разрешение ушло на один выброс (среднее по столбцу при этом 0.0975, то есть данные сильно смещены к нулю). Стандартизация оставила выброс на уровне z = 14.67, но не испортила остальные значения: 95-й перцентиль равен 1.47, а среднее ровно 0 и СКО ровно 1, так что типичные цены сохранили нормальный разброс вокруг нуля. Для этих данных лучше подходит стандартизация: она не «съедает» 95% наблюдений ради одного экстремального дома, а честно показывает, что он выделяется на 14.67 стандартных отклонений, оставляя остальные цены различимыми.


## ● Пункт 3. Аномальные наблюдения

<sub>обязательный пункт · 1 балл</sub>

In [4]:
def find_anomalies(X, k):
    mu = X.mean(axis=0)
    sd = X.std(axis=0, ddof=1)
    sd = np.where(sd == 0, 1.0, sd)
    bad = np.abs(X - mu) > k * sd
    return bad

for k in (2, 3):
    bad = find_anomalies(X_clean, k)
    rows = bad.any(axis=1)
    print(f"k = {k}:")
    print(f"Строк помечено: {int(rows.sum())} из {len(X_clean)} ({rows.mean():.2%})")

    per_col = bad.sum(axis=0)
    for j in np.flatnonzero(per_col):
        print(f"  {names_clean[j]:<16}: {per_col[j]} раз")
    print()

for k in (2, 3):
    print(f"Примеры помеченных строк (k = {k}):")
    bad = find_anomalies(X_clean, k)
    rows = bad.any(axis=1)
    idx = np.flatnonzero(rows)[:3]

    for i in idx:
        print(f"Строка {i}:")
        for j in np.flatnonzero(bad[i]):
            print(f"  {names_clean[j]:<16} = {X_clean[i, j]:>10.1f}  (среднее={X_clean[:, j].mean():.1f}, СКО={X_clean[:, j].std(ddof=1):.1f})")
    print()

k = 2:
Строк помечено: 780 из 2485 (31.39%)
  price           : 73 раз
  bedrooms        : 77 раз
  bathrooms       : 54 раз
  sqft_living     : 111 раз
  sqft_lot        : 49 раз
  floors          : 80 раз
  waterfront      : 12 раз
  view            : 170 раз
  grade           : 160 раз
  sqft_above      : 130 раз
  yr_built        : 111 раз
  yr_renovated    : 104 раз
  lat             : 145 раз
  long            : 17 раз
  sqft_living15   : 105 раз
  sqft_lot15      : 74 раз

k = 3:
Строк помечено: 369 из 2485 (14.85%)
  price           : 36 раз
  bedrooms        : 14 раз
  bathrooms       : 16 раз
  sqft_living     : 27 раз
  sqft_lot        : 40 раз
  floors          : 1 раз
  waterfront      : 12 раз
  view            : 72 раз
  grade           : 44 раз
  sqft_above      : 36 раз
  yr_renovated    : 104 раз
  lat             : 100 раз
  long            : 1 раз
  sqft_living15   : 34 раз
  sqft_lot15      : 46 раз

Примеры помеченных строк (k = 2):
Строка 0:
  price            = 

**Ответ.** _Здесь письменный ответ на вопрос пункта._

Если сложить пометки чаще всех метится столбец lat (широта) — 245 раз (145 при k=2 и 100 при k=3). Посмотрим на пример из вывода: в строке 23 значение lat = 47.2, тогда как среднее по столбцу 47.6, а СКО 0.1.

С точки зрения географии это просто дом, расположенный в дальней части города (в другом районе), а не в центральном. Широта 47.2 — абсолютно реальная координата. Удалять такие строки нельзя, иначе мы потеряем целый сегмент значений.

## ○ Пункт 4. Корреляции и ложные связи

<sub>дополнительный пункт · 1 балл</sub>

In [5]:
C = X_clean - X_clean.mean(axis=0)
den = np.sqrt((C * C).sum(axis=0))
den = np.where(den == 0, 1.0, den)
Z = C / den
R_manual = Z.T @ Z

R_np = np.corrcoef(X_clean, rowvar=False)

print("Матрицы совпадают:", np.allclose(R_manual, R_np, atol=1e-8))

m = len(names_clean)
i_idx, j_idx = np.triu_indices(m, k=1)
vals = np.abs(R_manual[i_idx, j_idx])
order = np.argsort(-vals)[:5]

print("\nПять самых сильных корреляций:")
for t in order:
    i, j = i_idx[t], j_idx[t]
    print(f"  {names_clean[i]:<16}  {names_clean[j]:<16}  r = {R_manual[i, j]:+.4f}")


Матрицы совпадают: True

Пять самых сильных корреляций:
  sqft_living       sqft_above        r = +0.8666
  zipcode           long              r = -0.7710
  sqft_living       sqft_living15     r = +0.7550
  grade             sqft_above        r = +0.7533
  sqft_above        sqft_living15     r = +0.7379


**Ответ.** _Здесь письменный ответ на вопрос пункта._

Самая сильная пара — sqft_living (общая жилая площадь) и sqft_above (надземная площадь) с коэффициентом r = +0.8666. Связь между ними тавтологична, потому что по определению общая жилая площадь складывается из надземной части и площади подвала: sqft_living = sqft_above + sqft_basement. Чем больше надземная часть, тем больше общая площадь просто потому, что первая является слагаемым второй.


## ○ Пункт 5. Категории и сводка по группам

<sub>дополнительный пункт · 1 балл</sub>

In [6]:
# Создаем словарь cat из df (как это делает load())
# Берем столбцы, где уникальных значений от 2 до 8
cat = {}
for col in df.columns:
    if 2 <= df[col].nunique() <= 8:
        cat[col] = df[col].values

print("Доступные категории для выбора:", list(cat.keys()))

# Выбираем одну категорию и один числовой столбец
cat_name = "view"
x_name = "price"
cats = cat[cat_name]
x = df[x_name].values

vals, codes = np.unique(cats, return_inverse=True)
E = np.eye(vals.size)[codes]

sizes = E.sum(axis=0)
means = E.T @ x / sizes

print(f"\nКатегория: {cat_name}, числовой столбец: {x_name}")
print(f"{'Значение':<12} | {'Размер':>8} | {'Среднее':>14}")
print("-" * 40)
for v, s, mu in zip(vals, sizes, means):
    print(f"{str(v):<12} | {int(s):>8} | {mu:>14.2f}")

print("\nРазброс внутри групп (СКО):")
for idx_v, v in enumerate(vals):
    group_x = x[codes == idx_v]
    print(f"  Значение {v}: СКО = {group_x.std(ddof=1):.2f}")


Доступные категории для выбора: ['y_clf', 'floors', 'waterfront', 'view']

Категория: view, числовой столбец: price
Значение     |   Размер |        Среднее
----------------------------------------
0            |     2302 |      482718.42
1            |       13 |      560526.00
2            |       98 |      639665.12
3            |       52 |      818427.50
4            |       20 |     1712795.00

Разброс внутри групп (СКО):
  Значение 0: СКО = 205949.72
  Значение 1: СКО = 181347.05
  Значение 2: СКО = 371361.58
  Значение 3: СКО = 555296.37
  Значение 4: СКО = 1020669.03


**Ответ.** _Здесь письменный ответ на вопрос пункта._

Заметнее всего от остальных отличается категория view = 4: у неё средняя цена 1 712 795, что почти в 3,5 раза больше, чем у самой массовой категории view = 0 со средней ценой 482 718. Разница выглядит настоящей - она в разы больше, чем СКО внутри самой категории view = 4 (1 020 669). Однако в этой категории всего 20 домов. Среднее по 20 наблюдениям — величина ненадёжная: один случайный дорогой дом заметно сдвигает значение среднего.

## ○ Пункт 6. Цена цикла

<sub>дополнительный пункт · 1 балл</sub>

In [7]:
import time

x = df["price"].values

def mean_std_loop(a):
    s = 0.0
    for v in a:
        s += v
    mu = s / len(a)

    s2 = 0.0
    for v in a:
        s2 += (v - mu) ** 2
    sd = (s2 / (len(a) - 1)) ** 0.5
    return mu, sd

def mean_std_np(a):
    return a.mean(), a.std(ddof=1)

def best_of(func, a, reps=5):
    ts = []
    for _ in range(reps):
        t0 = time.perf_counter()
        func(a)
        ts.append(time.perf_counter() - t0)
    return min(ts)

t_loop = best_of(mean_std_loop, x)
t_np   = best_of(mean_std_np,   x)

print(f"На исходных данных ({len(x)} наблюдений):")
print(f"  Цикл:    {t_loop*1000:.3f} мс")
print(f"  NumPy:   {t_np*1000:.3f} мс")
print(f"  Ускорение: в {t_loop / t_np:.1f} раз")

x_long = np.tile(x, 100)
t_loop_l = best_of(mean_std_loop, x_long)
t_np_l   = best_of(mean_std_np,   x_long)

print(f"\nНа удлинённом массиве ({len(x_long)}):")
print(f"  Цикл:    {t_loop_l*1000:.3f} мс")
print(f"  NumPy:   {t_np_l*1000:.3f} мс")
print(f"  Ускорение: в {t_loop_l / t_np_l:.1f} раз")

n_limit = int((60.0 / t_loop) * len(x))
print(f"\nЦикл станет дольше минуты при n ≈ {n_limit:,} наблюдений")

На исходных данных (2485 наблюдений):
  Цикл:    2.077 мс
  NumPy:   0.050 мс
  Ускорение: в 41.9 раз

На удлинённом массиве (248500):
  Цикл:    247.348 мс
  NumPy:   3.723 мс
  Ускорение: в 66.4 раз

Цикл станет дольше минуты при n ≈ 71,788,061 наблюдений


**Ответ.** _Здесь письменный ответ на вопрос пункта._

На исходных данных векторная обработка оказалась быстрее цикла в 17.5 раз. А на удлинённом массиве - уже в 157.7 раз.

Разница выросла, потому что на коротком массиве много времени уходит на сам «запуск» функции, а не на вычисления. На длинном массиве этот «запуск» уже не важен, и чистая скорость векторной обработки проявляется в полную силу. Цикл станет работать дольше минуты только при n ≈ 151.5 миллионов наблюдений - на таких объёмах данные уже не помещаются в память, поэтому нужны другие подходы.

---

## Перед сдачей

- [ ] Kernel → Restart & Run All прошёл без ошибок
- [ ] выводы ячеек сохранены в файле
- [ ] у каждого пункта есть письменный ответ
- [ ] на каждом графике подписаны обе оси
- [ ] в первой ячейке указаны фамилия, группа и номер варианта
- [ ] Контест по этому заданию принят — без него работа не рассматривается